In [1]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [2]:
from dotenv import load_dotenv
from google import genai
gemini_client = genai.Client()

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download the installer for your operating system (macOS, Windows, or Linux). For Linux, you can run:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

2.  **Start the model:** Open a terminal and type:
    ```bash
    ollama run llama3
    ```
    This will download the LLaMA 3 model and open a chat-like interface.

3.  **Test the server:** To verify the Ollama local server is running, use:
    ```bash
    curl http://localhost:11434
    ```

4.  **Use with Python (optional):** Install the Python client via `pip install ollama` to interact with the model through scripts.


In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The provided FAQ database does not contain information on how to run Ollama locally.


In [7]:
messages = [{'role': 'user', 'parts': [{'text': 'I just discovered the course. Can I join it?'}]}]

response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
)

response.text

'It sounds like you are excited to get started! However, **I am an AI, not a specific person or organization**, so I don’t know which course you are referring to.\n\nTo help you get the right answer, please check the following:\n\n1.  **Where did you find the course?** (Was it on a platform like Coursera, Udemy, edX, or a specific university/company website?)\n2.  **Is it a "Self-Paced" course?** If so, you can almost certainly join at any time.\n3.  **Is it a "Cohort-Based" course?** If it has set start and end dates, you might be too late for the current session, but you might be able to join a waitlist for the next one.\n\n**Here is what you should do next:**\n\n*   **Look for a "Join," "Enroll," or "Sign Up" button:** If the button is clickable, the course is likely open.\n*   **Check the Syllabus or FAQ page:** Often, these pages will state if late enrollment is permitted.\n*   **Contact the instructor or support team:** Look for a "Contact Us" or "Help" link on the course page. S

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
from google.genai import types

search_function = types.FunctionDeclaration(
    name='search',
    description='Search the FAQ database for entries matching the given query.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ['query']
    }
)

search_tool = types.Tool(function_declarations=[search_function])

In [11]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config={"tools": [search_tool]}
)

response.function_calls


[FunctionCall(
   args={
     'query': 'can I join the course late'
   },
   id='5efxrN6u',
   name='search'
 )]

In [12]:
len(response.function_calls)

1

In [13]:
call = response.function_calls[0]

In [14]:
call

FunctionCall(
  args={
    'query': 'can I join the course late'
  },
  id='5efxrN6u',
  name='search'
)

In [15]:
args = call.args
args

{'query': 'can I join the course late'}

In [16]:
call.name

'search'

In [17]:
results = search(**args)


In [18]:
function_response_part = types.Part(
    function_response=types.FunctionResponse(
        id=call.id,
        name=call.name,
        response={'result': results},
    )
)

In [19]:

messages.append(response.candidates[0].content)

messages.append(types.Content(role='user', parts=[function_response_part]))

In [20]:
messages

[{'role': 'user',
  'parts': [{'text': 'I just discovered the course. Can I join it?'}]},
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'can I join the course late'
         },
         id='5efxrN6u',
         name='search'
       ),
       thought_signature=b'\x124\n2\x01\x11M2\x0f\xe6\xdf\xa5\x17\xc6\xe8H\x0f\xd9x\x0eJ\x17\xf0D\xf8\x10\xc7\x188\xaa\x9c\xe6\x91\x06\xf6\x14es\xc7Oj^\x80\xbf\t:\x9c@-c\xbb\xdfT\x1e'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         id='5efxrN6u',
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
   role='user'
 )]

In [21]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config={"tools": [search_tool]}
)



In [22]:
print(response.text)


Yes, you can still join!

You are free to start the course whenever you want. Since all the materials (videos, notebooks, and homework instructions) are available on the [course GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp), you can work through them at your own pace.

However, please keep the following in mind:

*   **Certificates:** If you want to receive a certificate, you must submit your project while the course is still accepting submissions and participate in the peer-review process, which only happens while the course is running. Certificates are not awarded for self-paced study.
*   **Homework:** If you wish to submit homework, you must do so before the deadlines specified on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).
*   **Registration:** Registration is optional. You can start watching videos and working through the materials immediately without needing a confirmation email.

To get started, check out the [LLM Zoo

In [23]:
def calculate_gemini_cost(usage_metadata, input_price_per_million, output_price_per_million):
    prompt_tokens = getattr(usage_metadata, "prompt_token_count", 0) or 0
    candidate_tokens = getattr(usage_metadata, "candidates_token_count", 0) or 0
    thoughts_tokens = getattr(usage_metadata, "thoughts_token_count", 0) or 0

    billed_input_tokens = prompt_tokens
    billed_output_tokens = candidate_tokens + thoughts_tokens

    input_cost = billed_input_tokens / 1_000_000 * input_price_per_million
    output_cost = billed_output_tokens / 1_000_000 * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        'prompt_tokens': billed_input_tokens,
        'output_tokens': billed_output_tokens,
        'input_cost': input_cost,
        'output_cost': output_cost,
        'total_cost': total_cost,
    }

pricing_by_model = {
    'gemini-3.1-flash-lite': {'input': 0.30, 'output': 2.50},
}

usage_metadata = response.usage_metadata
model_name = getattr(response, 'model_version', 'gemini-3.1-flash-lite')
pricing = pricing_by_model.get(model_name, pricing_by_model['gemini-3.1-flash-lite'])

cost_breakdown = calculate_gemini_cost(
    usage_metadata,
    pricing['input'],
    pricing['output'],
)

print(f"Model: {model_name}")
print(f"Prompt tokens: {cost_breakdown['prompt_tokens']}")
print(f"Output tokens: {cost_breakdown['output_tokens']}")
print(f"Estimated input cost: ${cost_breakdown['input_cost']:.6f}")
print(f"Estimated output cost: ${cost_breakdown['output_cost']:.6f}")
print(f"Estimated total cost: ${cost_breakdown['total_cost']:.6f}")


Model: gemini-3.1-flash-lite
Prompt tokens: 822
Output tokens: 267
Estimated input cost: $0.000247
Estimated output cost: $0.000667
Estimated total cost: $0.000914


In [24]:
from google.genai import types

def make_call(call):
    # Gemini natively parses args into a dictionary, so json.loads is not needed
    args = call.args

    if call.name == 'search':
        result = search(**args)

    # Gemini expects a specific FunctionResponse object rather than a raw JSON string/dictionary
    return types.Part(
        function_response=types.FunctionResponse(
            id=call.id,
            name=call.name,
            response={'result': result},
        )
    )

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""
question = 'I just discovered the course. Can I join it?'
config = types.GenerateContentConfig(
    system_instruction=instructions,
    tools=[search_tool]
)

messages = [
    {'role': 'user', 'parts': [{'text': question}]}
]

In [26]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=messages,
    config=config
)


In [27]:
messages.append(response.candidates[0].content)

function_responses = []

if response.function_calls:
    for call in response.function_calls:
        print('function_call:', call.name, call.args)
        
        # Execute the function using your make_call method
        call_output = make_call(call)
        function_responses.append(call_output)
        
    # Append the results of the functions back to the conversation as a single user message
    messages.append(types.Content(role='user', parts=function_responses))
else:
    # If no functions were called, print the text response
    print('ASSISTANT:')
    print(response.text)

function_call: search {'query': 'can I join the course late registration enrollment policy'}


In [28]:
messages 

[{'role': 'user',
  'parts': [{'text': 'I just discovered the course. Can I join it?'}]},
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'can I join the course late registration enrollment policy'
         },
         id='XBYzBfx6',
         name='search'
       ),
       thought_signature=b'\x124\n2\x01\x11M2\x0f\x8f[t\x19p\xb2\xfe\xae\xf8\x0e<{\xf6M\xdf\xd1\xd9\xcd\xbb\xe1\xdb\xb1}\x8d\xbe\xc7\x90\x8b*CN\xcf\xf1\x00\x80+\x83\x9b\xb9\xe4Q\x0f\x05\x8a\x0f'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         id='XBYzBfx6',
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
 

In [29]:
messages = [
    {'role': 'user', 'parts': [{'text': question}]}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=messages,
        config=config # Uses the config we defined in the previous setup cell
    )

    if response.function_calls:
        has_function_calls = True
        
        # Append the model's function call request
        messages.append(response.candidates[0].content)
        
        function_responses = []
        
        # Loop through and execute each requested function
        for call in response.function_calls:
            print('function_call:', call.name, call.args)
            
            # Use the make_call function we converted earlier
            call_output = make_call(call)
            function_responses.append(call_output)
            
        # Append all function outputs as a single user message part
        messages.append(types.Content(role='user', parts=function_responses))
        
    else:
        # If no tools were called, print the final answer
        print('ASSISTANT:')
        print(response.text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {'query': 'can I join the course late registration enrollment'}
iteration #2...
ASSISTANT:
Yes, you can absolutely join the course! You don't need formal registration to start—you are welcome to begin learning and working through the materials immediately.

Here are a few key points to keep in mind:

*   **Getting Started:** You can begin by visiting the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp) and the [course documentation](https://datatalks.club/docs/courses/llm-zoomcamp/).
*   **Certificates:** If you are aiming for a certificate, please note that you must submit your project while the submission forms are still open. Certificates are only awarded to those who complete the course with a "live" cohort, as you will need to participate in peer-reviewing capstone projects.
*   **Homework:** You can submit homework through the course platform as long as the forms are still open. 

Are there any other areas of th

In [30]:
import time
def agent_loop(instructions, question, model="gemini-3.1-flash-lite") -> str:
    # Initialize the config with system instructions inside the function
    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[search_tool]
    )
    
    messages = [
        {'role': 'user', 'parts': [{'text': question}]}
    ]

    it = 1

    while True:
        time.sleep(4.5) # added the time.sleep to avoid rate limiting
        print(f'iteration #{it}...')
        has_function_calls = False

        response = gemini_client.models.generate_content(
            model=model,
            contents=messages,
            config=config
        )

        if response.function_calls:
            has_function_calls = True
            messages.append(response.candidates[0].content)
            
            function_responses = []
            
            for call in response.function_calls:
                print('function_call:', call.name, call.args)
                call_output = make_call(call)
                function_responses.append(call_output)
                
            messages.append(types.Content(role='user', parts=function_responses))
            
        else:
            last_answer = response.text
            print('ASSISTANT:')
            print(last_answer)
            
        it = it + 1
        if has_function_calls == False:
            break
            
    return last_answer

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [32]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': 'can I join the course late registration enrollment'}
iteration #2...
ASSISTANT:
Yes, you absolutely can! You are welcome to join and start the course at any time.

Here are a few important things to keep in mind:

*   **Registration:** You don't necessarily need to be officially registered to start. You can dive right into the materials, watch the videos, and work through the GitHub repositories immediately.
*   **Certificates:** If you are hoping to earn a certificate, please note that you must submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort (as you will need to participate in peer-reviews). You cannot earn a certificate if you follow the course entirely in self-paced mode after the cohort has finished.
*   **How to start:** You can find all the resources to get started in the [LLM Zoomcamp documentation](https://datatalks.club/docs/cours

In [33]:
result

'Yes, you absolutely can! You are welcome to join and start the course at any time.\n\nHere are a few important things to keep in mind:\n\n*   **Registration:** You don\'t necessarily need to be officially registered to start. You can dive right into the materials, watch the videos, and work through the GitHub repositories immediately.\n*   **Certificates:** If you are hoping to earn a certificate, please note that you must submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort (as you will need to participate in peer-reviews). You cannot earn a certificate if you follow the course entirely in self-paced mode after the cohort has finished.\n*   **How to start:** You can find all the resources to get started in the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nAre there any other area

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {'query': "what is queen's gambit"}
iteration #2...
ASSISTANT:
I have searched through the course FAQ, and there is no information regarding "the Queen's Gambit." This appears to be an off-topic question, as it does not relate to the course material or its logistics.

Are there any other areas of the course you would like to explore?


In [35]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {'query': 'run Olama locally'}
iteration #2...
function_call: search {'query': 'Ollama'}
iteration #3...
ASSISTANT:
To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    Visit [https://ollama.com/download](https://ollama.com/download) and choose the installer for your operating system:
    *   **macOS:** Download the `.pkg` and install it.
    *   **Windows:** Download the `.msi` and install it.
    *   **Linux:** Run the following command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Start and Run a Model:**
    Once installed, open your terminal and run a model (e.g., LLaMA 3):
    ```bash
    ollama run llama3
    ```
    This will download the model and open a chat interface in your terminal.

3.  **Verify the Server:**
    To ensure the local server is running correctly, you can test it with:
    ```bash
    curl http://localhost:11434
    ```
    If it's working, y

'To run Ollama locally, follow these steps:\n\n1.  **Install Ollama:**\n    Visit [https://ollama.com/download](https://ollama.com/download) and choose the installer for your operating system:\n    *   **macOS:** Download the `.pkg` and install it.\n    *   **Windows:** Download the `.msi` and install it.\n    *   **Linux:** Run the following command in your terminal:\n        ```bash\n        curl -fsSL https://ollama.com/install.sh | sh\n        ```\n\n2.  **Start and Run a Model:**\n    Once installed, open your terminal and run a model (e.g., LLaMA 3):\n    ```bash\n    ollama run llama3\n    ```\n    This will download the model and open a chat interface in your terminal.\n\n3.  **Verify the Server:**\n    To ensure the local server is running correctly, you can test it with:\n    ```bash\n    curl http://localhost:11434\n    ```\n    If it\'s working, you will receive a JSON response listing the available models.\n\n4.  **Using Ollama with Python:**\n    You can install the Pytho

In [36]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {'query': "what's queen gambit?"}
iteration #2...
ASSISTANT:
I am sorry, but I do not have any information about "Queen's Gambit" in the course FAQ. It appears this question may be off-topic regarding the course materials.

Are there any other areas related to the course that you would like to explore?


'I am sorry, but I do not have any information about "Queen\'s Gambit" in the course FAQ. It appears this question may be off-topic regarding the course materials.\n\nAre there any other areas related to the course that you would like to explore?'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")
# ALL good till here

iteration #1...
function_call: search {'query': 'what is queen gambit'}
iteration #2...
ASSISTANT:
I could not find any information about "Queen's Gambit" in the course FAQ database. It appears that this question is not related to the course content.

Are there any other areas regarding the course or its logistics that you would like to explore?


'I could not find any information about "Queen\'s Gambit" in the course FAQ database. It appears that this question is not related to the course content.\n\nAre there any other areas regarding the course or its logistics that you would like to explore?'

In [63]:

# Cleanup note
# We are not reproducing the workshop's toyaikit runner layer here.
# That version adds an extra abstraction on top of the direct Gemini API,
# depends on package-specific runner classes, and is beyond the minimal scope
# of this notebook.
# Instead, we keep the manual Gemini tool-calling flow above, which shows the
# same idea with less complexity and no extra dependency on external runner files.